# BÁO CÁO DỰ ÁN: XÂY DỰNG MÔ HÌNH HỌC MÁY DỰ ĐOÁN LƯƠNG CHÀO TUYỂN CNTT VIỆT NAM

**Dự án:** `theihoz/ML` - Hệ thống phân tích thị trường việc làm và dự đoán lương lập trình viên.  
**Mô hình sử dụng trong dự án:** **Ridge Regression** (Champion Model) và **Linear Regression (OLS)**.  
**Sản phẩm bàn giao:** File Jupyter Notebook giải đáp toàn diện các câu hỏi đề tài dựa trên mã nguồn, dữ liệu thực tế và kết quả huấn luyện đã lưu trong repository.

---

### TỔNG QUAN BÀI TOÁN
Dự án tập trung vào bài toán cốt lõi: **Dự đoán mức lương chào tuyển ước tính (VND/tháng) của các vị trí CNTT tại Việt Nam** dựa trên vai trò nghề nghiệp (`role`), địa điểm làm việc (`locations`), hình thức làm việc (`work_modes`), kỹ năng công nghệ (`skills`) và kinh nghiệm. 

Toàn bộ quy trình được thực hiện theo quy chuẩn kỹ thuật nghiêm ngặt (ADR 0001 - ADR 0012):
- Không sử dụng mô hình nạp sẵn (pretrained), không fine-tune mô hình ngoài hay dịch vụ đám mây bên thứ ba.
- Hai mô hình được tự khởi tạo và huấn luyện từ đầu là **Ridge Regression** và **Linear Regression (OLS)**.
- Quá trình tiền xử lý, huấn luyện và đánh giá được kiểm duyệt khắt khe: chia theo nhóm công ty (Company-held-out split) để chống rò rỉ dữ liệu (Data Leakage); khóa trước bộ quy tắc (Policy Lock); chọn mô hình hoàn toàn dựa trên tập Validation và giữ Test độc lập.
- Mô hình chiến thắng (Ridge Regression) được đóng gói thành artifact và triển khai vào ứng dụng web giao diện tiếng Việt bằng **Streamlit** (`job_market/app.py`).


# 1. LỰA CHỌN DATASET VÀ TRẢ LỜI CÁC CÂU HỎI

### 1.1. Dataset lấy từ đâu? (Link / nền tảng / trang web...)

Dự án khai thác và kiểm soát các nguồn dữ liệu công khai trên nền tảng **Kaggle** theo tài liệu `job_market/src/career_market/sources.json` và quy chuẩn kiểm duyệt nhãn ADR 0002, ADR 0010:

#### 1. Nguồn huấn luyện ML cốt lõi: `vitthnhnguyn138/vietnam-it-job-posts` (v1)
* **Nền tảng / Trang web:** Kaggle.
* **Tác giả / Publisher:** Việt Thành Nguyễn (`vitthnhnguyn138`).
* **Đường dẫn (URL):** [https://www.kaggle.com/datasets/vitthnhnguyn138/vietnam-it-job-posts](https://www.kaggle.com/datasets/vitthnhnguyn138/vietnam-it-job-posts)
* **Giấy phép (License):** MIT License (theo metadata chính thức của Kaggle).
* **Tệp thô trong dự án:** `job_market/datasets/raw/vietnam_it_job_posts_v1.csv` (Mã băm SHA-256: `d201d539d1bbc79a54a369eafc94d85efbb10bdece9f6ac80b9e5e4c9a0bc30c`).
* **Mô tả nguồn gốc:** Tác giả thu thập từ hơn 2.500 tin tuyển dụng CNTT tại Hà Nội và toàn quốc qua 5 cổng việc làm: VietnamWorks, ITviec, YBox, TopCV và CareerViet.

#### 2. Các nguồn khảo sát và đối sánh bổ trợ trong dự án:
* **Nguồn `jobs-2023` (CareerBuilder):** Do Nguyễn Trần Hữu Thịnh (`lucaznguyenofficial`) công bố trên Kaggle ([Link Kaggle](https://www.kaggle.com/datasets/lucaznguyenofficial/vietnamese-job-posting-vietnamese-version)), giấy phép CC0: Public Domain. Tệp lưu trữ: `job_market/datasets/raw/vietnamese-job-posting.csv`. Chứa toàn văn Mô tả công việc (Job Description) và Yêu cầu công việc (Job Requirements).
* **Nguồn `jobs-2026` (TopCV):** Do Châu Gia Bảo (`baocgb`) công bố trên Kaggle ([Link Kaggle](https://www.kaggle.com/datasets/baocgb/vietnam-it-jobs-raw-data-from-topcv-2026)), giấy phép CC BY 4.0. Tệp lưu trữ: `job_market/datasets/raw/topcv_jobs.csv`.

---

### 1.2. Có bao nhiêu mẫu dữ liệu?

Dự án phân biệt rõ ràng các tầng dữ liệu từ dữ liệu thô đến tập nhãn chuẩn hóa đưa vào huấn luyện mô hình:
* **Dữ liệu thô từ Kaggle (`vietnam_it_job_posts_v1.csv`):** **6.779 dòng**.
* **Dòng có cấu trúc khoảng lương tháng (`tr-tr ₫/tháng`):** **336 dòng** được trích xuất vào sổ cái kiểm duyệt (`review-ledger.json`).
* **Tập dữ liệu chuẩn hóa phục vụ phân tích EDA (`eda.csv`):** **370 mẫu tin tuyển dụng** thuộc 7 nhóm nghề CNTT tại Hà Nội, TP.HCM và Đà Nẵng.
* **Tập nhãn kiểm duyệt khắt khe đưa vào huấn luyện mô hình (`accepted.csv`):** **14 thực thể tuyển dụng (entity)** đến từ **11 công ty độc lập**. Đây là tập nhãn đạt đầy đủ tiêu chí hợp đồng (nhãn rõ ràng, đơn vị VND/tháng, vai trò và địa điểm rõ ràng, không mâu thuẫn phiên bản).

---

### 1.3. Có những thuộc tính / đặc trưng nào? Mô tả từng thuộc tính / đặc trưng

#### A. Các thuộc tính thô từ nguồn Kaggle:
* `title`: Tiêu đề tin tuyển dụng gốc (ví dụ: `Junior AI/NLP Engineer`, `Backend Developer`, `Devops & SRE - Level: Senior`).
* `company`: Tên doanh nghiệp đăng tuyển (ví dụ: Sunteco, Consortio Việt Nam, Socotec, Thư Viện Pháp Luật...).
* `location`: Tỉnh/thành phố làm việc ghi trong tin.
* `salary`: Chuỗi văn bản thô thể hiện mức lương (ví dụ: `20tr-30tr ₫/tháng`, `14tr-25tr ₫/tháng`).
* `experience`: Yêu cầu số năm kinh nghiệm theo văn bản gốc.
* `skills`: Mảng từ khóa kỹ năng công nghệ được trích xuất.
* `source_platform`, `source_url`: Nền tảng gốc (vietnamworks, itviec...) và URL định danh tin.

#### B. Các thuộc tính chuẩn hóa đưa vào mô hình học máy (Feature Schema trong `training.py`):
Theo nguyên tắc **Feature Allowlist** nghiêm ngặt (chống rò rỉ dữ liệu), chỉ những thuộc tính người dùng được phép nhập trong form dự đoán mới được đưa vào ma trận $X$:

| STT | Tên thuộc tính | Kiểu dữ liệu | Mô tả chi tiết | Các giá trị thực tế trong dự án |
|:---:|---|:---:|---|---|
| 1 | `role` | Categorical (One-hot) | Vai trò nghề nghiệp của tin tuyển dụng, phân thành 7 nhóm công việc chuẩn theo `GLOSSARY.md`. | `Backend`, `Frontend`, `Fullstack`, `Mobile`, `Data/ML`, `DevOps`, `QA Automation` |
| 2 | `locations` | Multi-hot / Binary | Địa điểm làm việc được chuẩn hóa về danh mục chuẩn theo ADR 0002. | `HCM`, `HN`, `Đà Nẵng` |
| 3 | `work_modes` | Binary indicator | Hình thức làm việc từ xa, độc lập với địa điểm vật lý. | `Remote` (1.0 nếu có, 0.0 nếu không) |
| 4 | `skills` | Multi-hot / Binary | Kỹ năng công nghệ có chứng cứ trích xuất từ tiêu đề tin tuyển dụng. | `Python`, `Java`, `C#`, `.NET`, `React`... |
| 5 | `experience_kind` | Categorical | Phân loại trạng thái kinh nghiệm của tin. | `missing` (do 14 mẫu chưa xác minh được số năm) |
| 6 | `experience_min_missing` | Binary (0.0 / 1.0) | Cờ chỉ báo thiếu số năm kinh nghiệm tối thiểu. | 1.0 khi thiếu, 0.0 khi có số năm |
| 7 | `experience_max_missing` | Binary (0.0 / 1.0) | Cờ chỉ báo thiếu số năm kinh nghiệm tối đa. | 1.0 khi thiếu, 0.0 khi có số năm |
| 8 | `salary_target_vnd_month` | Numerical (Float) | **Biến mục tiêu ($y$):** Mức lương chào tuyển trung điểm tháng, tính bằng **VND/tháng**. | Trung bình cộng cận dưới và cận trên: $\frac{\text{min} + \text{max}}{2}$ |

---

### 1.4. Giải thích tại sao dataset phù hợp với bài toán
1. **Có chứng cứ xác thực về mức lương thị trường:** Khác với các bộ dữ liệu khảo sát tự điền, dataset này thu thập trực tiếp các tin tuyển dụng đang chào mời trên thị trường với dải lương bằng số cụ thể (ví dụ `20tr-30tr ₫/tháng` $\rightarrow$ lương trung điểm là 25.000.000 VND/tháng).
2. **Đồng nhất về tiền tệ và chu kỳ trả lương:** Toàn bộ tập nhãn huấn luyện được kiểm duyệt đồng nhất theo đơn vị **VND/tháng**, loại bỏ các tin ngoại tệ USD hay tin ghi lương theo năm, giúp mô hình hồi quy học được đúng thang đo tiền tệ tại Việt Nam mà không bị sai lệch tỷ giá.
3. **Phù hợp với 7 nhóm vai trò và 3 thành phố trọng điểm:** Dataset bao phủ các vị trí kỹ thuật cốt lõi tại Hà Nội, TP. Hồ Chí Minh và Đà Nẵng, phù hợp trực tiếp với giao diện chọn nghề và địa điểm của ứng dụng web Streamlit.

---

### 1.5. Nêu ra hạn chế (hoặc giới hạn) của dataset
1. **Quy mô tập huấn luyện thăm dò còn nhỏ (Small Exploratory Cohort):**
   - Chỉ có **14 thực thể từ 11 công ty** đáp ứng đầy đủ tiêu chí kiểm duyệt nhãn nghiêm ngặt. Số lượng mẫu ít khiến sai số kiểm thử còn lớn và một số nhóm nghề trong tập Train chỉ có 1–2 mẫu (thậm chí chưa có mẫu Fullstack trong Train).
2. **Kỹ năng mới chỉ trích xuất từ tiêu đề (Title-only Skills):**
   - Trong 14 tin huấn luyện, chỉ có 3 tin có kỹ năng trích xuất được từ tiêu đề (như Java, Python, C#); các tin còn lại chưa khai thác được kỹ năng chi tiết do nguồn thô thiếu Job Description đầy đủ.
3. **Chưa xác minh được đơn vị số năm kinh nghiệm (`numeric_experience_units_unconfirmed`):**
   - Toàn bộ 14 nhãn đều chưa có số năm kinh nghiệm được kiểm chứng đơn vị rõ ràng trong văn bản nguồn, do đó mô hình trong đợt huấn luyện này chưa học được ảnh hưởng định lượng độc lập của số năm kinh nghiệm.
4. **Chưa phân biệt Gross/Net và cấu phần lương:**
   - Đa số tin tuyển dụng không ghi rõ mức lương chào tuyển là Gross hay Net (thuộc nhóm `unknown` theo ADR 0006) và không tách bạch lương cơ bản với phụ cấp/thưởng (`salary_components_unspecified`).


# 2. TIỀN XỬ LÝ DỮ LIỆU (SỬ DỤNG NUMPY, PANDAS)

### 2.1. Kiểm tra kích thước dữ liệu
* Kích thước dữ liệu thô: `(6779, 19)` — 6.779 hàng, 19 cột.
* Kích thước tập dữ liệu EDA chuẩn hóa: `(370, 54)` — 370 dòng tin CNTT đã được gán vai trò, địa điểm và kỹ năng.
* Kích thước tập nhãn kiểm duyệt trước fit: `(14, 35)` — 14 thực thể tuyển dụng hợp lệ.

---

### 2.2. Xác định các thuộc tính có giá trị bị thiếu (Missing Values)
1. **Thuộc tính kinh nghiệm (`experience_min`, `experience_max`):**
   - Trong 14 mẫu huấn luyện, 100% thiếu thông tin số năm kinh nghiệm có bằng chứng rõ ràng.
   - **Nguyên tắc xử lý của dự án:** Tuyệt đối **không điền số giả định** (như số 0, trung bình hay trung vị) vào số năm kinh nghiệm. Thay vào đó, sử dụng các cờ chỉ báo thiếu đặc trưng:
     * `experience_min_missing = 1.0`
     * `experience_max_missing = 1.0`
     * `experience_kind = "missing"`
   - Thiết kế này giúp mô hình nhận biết trạng thái thiếu thông tin mà không làm méo mó trọng số hồi quy.
2. **Thuộc tính kỹ năng (`skills`):**
   - Các tin không có từ khóa kỹ năng trong tiêu đề được biểu diễn dưới dạng danh sách rỗng `[]`, khi qua `DictVectorizer` sẽ nhận giá trị 0.0 đối với tất cả các cột kỹ năng.

---

### 2.3. Kiểm tra dữ liệu trùng lặp (Duplicates)
1. **Kiểm tra trùng lặp tin đăng lại (Reposts):**
   - Phát hiện các tin đăng lại của cùng công ty (ví dụ: Sunteco đăng nhiều tin Backend cùng dải lương ở các thời điểm khác nhau, hoặc Thư Viện Pháp Luật đăng trùng vị trí AI Architect).
   - Dự án xây dựng bảng `repost_groups.json`: các tin trùng lặp chỉ giữ lại 1 thực thể đại diện trong tập nhãn, đồng thời toàn bộ các tin thuộc cùng công ty Sunteco hay Thư Viện Pháp Luật đều được gom chung vào một nhóm duy nhất để phân chia Train/Test.
2. **Kiểm tra trùng lặp đơn vị tuyển dụng (Company Overlap):**
   - 14 mẫu thuộc về **11 nhóm công ty**. Phép chia tập được thực hiện theo nhóm công ty (Group Split) để đảm bảo không có công ty nào vừa xuất hiện ở Train vừa xuất hiện ở Test.

---

### 2.4. Xác định các thuộc tính không cần thiết và nêu lý do loại bỏ

Quy tắc **Feature Allowlist** nghiêm ngặt loại bỏ các thuộc tính sau khỏi ma trận đặc trưng $X$:

| Thuộc tính loại bỏ | Lý do kỹ thuật & nguyên tắc chống rò rỉ (Leakage) |
|---|---|
| `salary`, `salary_min`, `salary_max`, `salary_target_vnd_month` | **Rò rỉ mục tiêu (Target Leakage):** Đây là thuộc tính đích cần dự đoán, tuyệt đối không đưa vào $X$. |
| `title`, `job_description` nguyên văn | **Tránh rò rỉ lương ngầm:** Tiêu đề hoặc nội dung văn bản có thể chứa các con số tiền lương hoặc cụm từ tiết lộ thu nhập (ví dụ: *"Junior AI 20-30 triệu"*). Chỉ trích xuất vai trò và kỹ năng chuẩn hóa vào $X$. |
| `id`, `job_hash`, `source_url`, `url` | **Định danh kỹ thuật:** Không mang ý nghĩa năng lực nghề nghiệp. |
| `company` | **Chống học vẹt chính sách doanh nghiệp:** Tên công ty chỉ được dùng làm biến phân nhóm (Group Key) khi chia tập, không đưa vào $X$ để mô hình học đặc trưng tổng quát của nghề và địa điểm chứ không học riêng từng công ty. |
| `posted_at`, `crawl_time` | **Thời gian ngắn hạn:** Dấu thời gian thu thập không phản ánh năng lực ứng viên. |

*Ma trận đặc trưng $X$ chỉ chứa 12 đặc trưng hợp lệ (sau khi DictVectorizer mã hóa từ tập Train):*
`experience_kind=missing`, `experience_max_missing`, `experience_min_missing`, `locations=HCM`, `locations=HN`, `role=Backend`, `role=Data/ML`, `role=DevOps`, `role=Frontend`, `skills=C#`, `skills=Java`, `skills=Python`.

---

### 2.5. Thực hiện tối thiểu 02 biểu đồ trực quan hóa có ý nghĩa đối với bài toán

Trong module `job_market/src/career_market/salary_visuals.py` và Dashboard Tổng quan Streamlit (`job_market/pages/overview.py`), dự án triển khai 3 biểu đồ trực quan hóa cốt lõi:

#### Biểu đồ 1: Phân bố mức lương chào tuyển theo 7 nhóm vai trò CNTT (Salary by Roles)
* **Ý nghĩa:** Trực quan hóa mặt bằng thu nhập giữa các chuyên ngành công nghệ để phục vụ mô hình hồi quy.
* **Số liệu phân tích thực tế:**
  - **Data/ML:** Nhóm có dải lương cao nhất thị trường, dao động từ **25 triệu đến trên 110 triệu VND/tháng** (các vị trí Senior AI Architect đạt mức 110 triệu VND/tháng).
  - **DevOps / SRE:** Đạt mức lương trung bình cao từ **25 triệu đến 90 triệu VND/tháng** (Senior DevOps đạt mức 90 triệu VND/tháng).
  - **Fullstack:** Trung điểm khoảng **70 - 85 triệu VND/tháng** cho cấp Senior.
  - **Backend Developer:** Phổ biến từ **17 triệu đến 25 triệu VND/tháng** cho cấp Middle/Junior.
  - **Frontend Developer:** Dao động trong khoảng **14 triệu đến 21 triệu VND/tháng**.

#### Biểu đồ 2: So sánh mức lương chào tuyển giữa 3 trung tâm kinh tế (HN, HCM, Đà Nẵng)
* **Ý nghĩa:** Đánh giá sự chênh lệch thu nhập địa phương để đưa vào hệ số hồi quy của biến `locations`.
* **Số liệu phân tích thực tế:**
  - **TP. Hồ Chí Minh:** Có mức trần lương cao nhất (đạt mức bình quân trung điểm 60–80 triệu VND/tháng ở các vị trí Senior AI/DevOps trong tập nhãn).
  - **Hà Nội:** Có số lượng tin đông đảo nhất, mức lương phân bố ổn định trong khoảng 20–40 triệu VND/tháng.
  - **Đà Nẵng:** Mức lương chào tuyển tập trung ở dải 25–35 triệu VND/tháng.

#### Biểu đồ 3: Biểu đồ phân tán Kinh nghiệm - Thu nhập (Experience vs Income)
* **Ý nghĩa:** Khảo sát mối tương quan giữa số năm kinh nghiệm và mức thu nhập chào tuyển trong tập dữ liệu tổng quan.
* **Số liệu:** Mức lương tăng trưởng rõ rệt từ mức 10–18 triệu (0–1 năm), 18–35 triệu (2–4 năm), đến 40–80+ triệu VND/tháng cho các vị trí yêu cầu từ 5 năm kinh nghiệm trở lên.


# 3. LỰA CHỌN MÔ HÌNH AI / ML

### 3.1. Xác định loại bài toán ML
* **Loại bài toán:** **Học có giám sát - Hồi quy (Supervised Regression)**.
* **Biến mục tiêu ($y$):** Mức lương chào tuyển trung điểm (`salary_target_vnd_month`), đơn vị tính: **VND/tháng**.
* **Đặc trưng đầu vào ($X$):** Biểu diễn dạng từ điển đặc trưng thưa gồm vai trò (`role`), địa điểm (`locations`), kỹ năng (`skills`) và cờ kinh nghiệm.

---

### 3.2. Lựa chọn thuật toán và giải thích lý do lựa chọn

Dự án triển khai và so sánh **2 thuật toán học máy chính** cùng **1 baseline đối chứng**:
1. **Ridge Regression (Hồi quy Ridge với điều chuẩn L2)**.
2. **Linear Regression (Hồi quy tuyến tính OLS thông thường)**.
3. **Train Median Baseline (Mô hình cơ sở trung vị tập train)**.

#### Quy trình Pipeline Scikit-learn (trong `training.py`):
$$\mathbf{x} \xrightarrow{\text{DictVectorizer(sparse=False)}} \text{Feature Matrix} \xrightarrow{\text{StandardScaler()}} \text{Scaled Features} \xrightarrow{\text{Ridge / LinearRegression}} \hat{y} \text{ (triệu VND)}$$

#### Các quyết định kỹ thuật quan trọng:
* **Ổn định số học:** Mức lương chào tuyển tính bằng VND thường là những con số hàng chục triệu (ví dụ 25.000.000). Nếu đưa trực tiếp số tiền này vào mô hình tuyến tính, gradient và hàm mất mát bình phương sẽ bùng nổ. Do đó, biến mục tiêu $y$ được **chia cho 1.000.000** khi fit và nhân lại 1.000.000 khi suy luận/tính metric.
* **Độc lập tiền xử lý:** Cả `DictVectorizer` và `StandardScaler` **chỉ được fit trên tập Train**, tuyệt đối không fit lại trên Validation hay Test để tránh rò rỉ phân phối dữ liệu (Data Leakage).
* **Lý do lựa chọn Ridge và Linear Regression:**
  - Không sử dụng mô hình nạp sẵn (pretrained) hay thư viện ngoài theo cam kết đề tài.
  - Khi số chiều đặc trưng (12 chiều) tương đương với số mẫu huấn luyện (10 mẫu train), mô hình hồi quy tuyến tính cổ điển (OLS) rất dễ bị đa cộng tuyến và quá khớp. **Ridge Regression** với thành phần phạt $L_2$ chính là giải pháp toán học chuẩn mực nhất để co nhỏ các trọng số (shrinkage), giảm phương sai mô hình và mang lại khả năng tổng quát hóa ổn định.

---

### 3.3. Trình bày ít nhất 02 thuật toán (mô hình) có thể giải quyết bài toán

#### 1. Mô hình Hồi quy Ridge (Ridge Regression)
- Là mô hình hồi quy tuyến tính có bổ sung thành phần điều chuẩn $L_2$ (Tikhonov Regularization) vào hàm mất mát bình phương sai số:
  $$\mathcal{L}_{\text{Ridge}}(\mathbf{w}, b) = \frac{1}{N} \sum_{i=1}^{N} \left( y_i - (\mathbf{w}^T \mathbf{x}_i + b) \right)^2 + \alpha \sum_{j=1}^{d} w_j^2$$
- Nghiệm giải tích dạng đóng:
  $$\mathbf{w} = (\mathbf{X}^T \mathbf{X} + \alpha \mathbf{I})^{-1} \mathbf{X}^T \mathbf{y}$$
- **Cấu hình siêu tham số cố định trong dự án:** `alpha = 10.0`, giải thuật `solver = 'svd'`. Tham số này được khóa cứng trong `training-policy-v1.json` trước khi fit, không dò tìm tham số trên tập Test.

#### 2. Mô hình Hồi quy tuyến tính cổ điển (Linear Regression - OLS)
- Tìm vector trọng số $\mathbf{w}$ và hệ số chặn $b$ tối thiểu hóa tổng bình phương sai số thông thường (Ordinary Least Squares):
  $$\mathcal{L}_{\text{OLS}}(\mathbf{w}, b) = \frac{1}{N} \sum_{i=1}^{N} \left( y_i - (\mathbf{w}^T \mathbf{x}_i + b) \right)^2$$
- Nghiệm giải tích dạng đóng:
  $$\mathbf{w} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$
- Được sử dụng làm mô hình đối sánh trực tiếp để kiểm chứng hiệu quả của việc điều chuẩn $L_2$.

---

### 3.4. Giải thích ưu điểm và hạn chế của từng thuật toán

| Thuật toán | Ưu điểm (Pros) | Hạn chế (Cons) |
|---|---|---|
| **Ridge Regression** | • Thành phần phạt $\alpha \|\mathbf{w}\|^2$ giải quyết triệt để vấn đề ma trận $\mathbf{X}^T \mathbf{X}$ suy biến hoặc có định thức gần bằng 0 khi số mẫu ít ($N \approx d$).<br>• Kiểm soát rất tốt hiện tượng đa cộng tuyến giữa các kỹ năng và địa điểm.<br>• Giảm phương sai (variance), hạn chế tối đa nguy cơ quá khớp (overfitting) trên tập dữ liệu nhỏ.<br>• Bộ giải SVD mang lại độ ổn định số học tuyệt đối. | • Hệ số hồi quy bị co nhỏ về gần 0 nhưng không bằng 0 tuyệt đối (không loại bỏ đặc trưng như Lasso).<br>• Cần chuẩn hóa thang đo (`StandardScaler`) trước khi huấn luyện để thành phần phạt tác động công bằng lên mọi đặc trưng. |
| **Linear Regression (OLS)** | • Mô hình toán học trong sáng, tốc độ tính toán tức thì.<br>• Trọng số thể hiện trực tiếp mức đóng góp biên không chệch (unbiased) của từng đặc trưng lên mức lương dự đoán. | • Rất dễ bị bất ổn định số học và quá khớp nặng nề khi dữ liệu có ít mẫu và nhiều đặc trưng.<br>• Cực kỳ nhạy cảm với các điểm ngoại lai (outliers) trong tập dữ liệu lương. |
| **Train Median Baseline** | • Cực kỳ bền vững với outlier, cung cấp mốc tham chiếu thực tế mà mọi mô hình thực phải vượt qua. | • Không học được mối quan hệ giữa các đặc trưng (nghề, kỹ năng, địa điểm) với mức lương. |

---

### 3.5. Trình bày cách chia dữ liệu thành Train / Test

#### Phương pháp áp dụng: **Chia theo nhóm công ty độc lập (Company-held-out Split)**
* **Khóa phân nhóm:** Thuộc tính `company` đã chuẩn hóa (và các nhóm tin đăng lại `repost_group_id`).
* **Tỷ lệ phân chia mục tiêu:** **70% Train / 15% Validation / 15% Test** với seed cố định `seed = 42`.
* **Kết quả phân chia thực tế (trong `split.json`):**
  - **Tập Train (Huấn luyện):** **10 tin tuyển dụng**, thuộc **7 nhóm công ty**.
  - **Tập Validation (Kiểm định):** **2 tin tuyển dụng**, thuộc **2 nhóm công ty độc lập**.
  - **Tập Test (Kiểm thử):** **2 tin tuyển dụng**, thuộc **2 nhóm công ty độc lập**.
  - **Giao cắt công ty giữa các tập:** **Hoàn toàn bằng 0** (không có bất kỳ công ty nào xuất hiện ở nhiều hơn một tập).

#### Quy trình đánh giá và khóa chính sách (Policy Lock):
1. **Khóa trước khi fit:** File cấu hình chính sách `training-policy-v1.json` và tệp nhãn được băm SHA-256 (`33cadf3...`) và khóa cứng trước lần fit đầu tiên.
2. **Chọn mô hình bằng Validation:** Mô hình tốt nhất (Champion Model) được lựa chọn **hoàn toàn dựa trên chỉ số Validation MAE**. Quyết định chọn mô hình được ghi nhận vào file `selection.json` TRƯỚC KHI thực hiện bất kỳ dự đoán nào trên tập Test.
3. **Bảo toàn tính độc lập của Test:** Tập Test chỉ được mở ra đánh giá nghiệm thu đúng 1 lần duy nhất sau khi đã chọn xong mô hình. Tuyệt đối không dùng kết quả Test để chọn lại mô hình, không chỉnh lại `seed` hay thay đổi `alpha` sau khi nhìn thấy Test.

---

### 3.6. Đề xuất metric đánh giá và giải thích lý do

#### 1. Metric chính: **MAE (Mean Absolute Error - Sai số tuyệt đối trung bình)**
$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$
* **Đơn vị đo:** VND/tháng (hoặc triệu VND/tháng).
* **Lý do lựa chọn:**
  - MAE đo lường trực tiếp mức độ sai lệch tiền mặt trung bình trên mỗi tin tuyển dụng (ví dụ: dự đoán lệch trung bình 10,5 triệu đồng).
  - Khác với MSE/RMSE, MAE không bình phương sai số nên không phạt quá nặng các giá trị lương cao tự nhiên của thị trường (outliers), phù hợp với bản chất phân phối lương CNTT thường bị lệch phải.
  - Mang ý nghĩa kinh tế và trực giác cao đối với người dùng cuối khi xem xét mức lương chào tuyển.

#### 2. Metric phụ: **RMSE (Root Mean Squared Error)**
$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$
* Đo lường phương sai sai số, giúp phát hiện các dự đoán có độ sai lệch đột biến.

#### 3. Tiêu chí phát hành mô hình (Release Criterion):
Mô hình chỉ được coi là đạt yêu cầu và được phát hành (Promoted) khi thỏa mãn đồng thời hai điều kiện:
$$\text{MAE}_{\text{Model}}(\text{Validation}) < \text{MAE}_{\text{Baseline}}(\text{Validation}) \quad \text{và} \quad \text{MAE}_{\text{Model}}(\text{Test}) < \text{MAE}_{\text{Baseline}}(\text{Test})$$

---

### 3.7. Kết quả thực nghiệm và Lựa chọn mô hình trong dự án (Run 7137291a5d16b71f1861)

Bảng kết quả trích xuất trực tiếp từ file báo cáo nghiệm thu [`report.json`](file:///Users/scara2906/Downloads/ML/job_market/artifacts/runs/7137291a5d16b71f1861/report.json):

| Mô hình | Validation MAE (triệu VND) | Validation RMSE (triệu VND) | Test MAE (triệu VND) | Test RMSE (triệu VND) | Quyết định phát hành |
|---|:---:|:---:|:---:|:---:|---|
| **Ridge Regression** | **10,50** | **11,62** | **21,98** | **22,20** | **CHỌN (Champion Model - Promoted)** |
| **Linear Regression** | 13,58 | 13,72 | 17,58 | 17,66 | Loại ở vòng Validation |
| **Train Median Baseline** | 11,25 | 11,86 | 32,50 | 42,57 | Bị Ridge đánh bại trên cả 2 tập |

* **Phân tích quyết định:**
  - Trên tập **Validation**: Ridge Regression đạt MAE là **10,50 triệu VND/tháng**, thấp hơn rõ rệt so với Linear Regression (13,58 triệu) và Baseline (11,25 triệu). Do đó, **Ridge Regression chính thức được chọn làm Champion Model**.
  - Trên tập **Test**: Ridge đạt MAE là **21,98 triệu VND/tháng**, đánh bại Train Median Baseline (32,50 triệu VND/tháng). Do đó mô hình đạt tiêu chí phát hành (`promoted: true`, `release_reason: "beats_baseline_on_validation_and_test"`).
  - Dù trên tập Test, Linear Regression có MAE thấp hơn (17,58 triệu), nhưng theo nguyên tắc khoa học đã khóa cứng trước fit: **không được phép đổi sang Linear Regression sau khi nhìn thấy kết quả Test** (tránh Data Peeking).
  - Pipeline Ridge Regression đã được lưu thành artifact [`Ridge.joblib`](file:///Users/scara2906/Downloads/ML/job_market/artifacts/runs/7137291a5d16b71f1861/Ridge.joblib) và được nạp trực tiếp vào giao diện ứng dụng Streamlit tại [`job_market/app.py`](file:///Users/scara2906/Downloads/ML/job_market/app.py).


# 4. TỔNG KẾT VÀ BÀN GIAO SẢN PHẨM DỰ ÁN

### 4.1. Bảng đối chiếu câu hỏi đề bài và giải pháp thực tế trong dự án

| Nội dung đề bài | Giải pháp thực tế trong repository `theihoz/ML` | Tệp mã nguồn / Tài liệu tương ứng |
|---|---|---|
| **Nguồn Dataset** | Kaggle (`vitthnhnguyn138/vietnam-it-job-posts` v1, MIT License, SHA-256 `d201d53...`) cùng TopCV và CareerBuilder | `job_market/src/career_market/sources.json`<br>`docs/adr/0002-new-datasets-and-canonical-labels.md` |
| **Số lượng mẫu** | 6.779 dòng thô; 370 dòng EDA; 14 thực thể kiểm duyệt từ 11 công ty | `job_market/datasets/labels/2026-10-30/accepted.csv` |
| **Đặc trưng dữ liệu** | 7 vai trò CNTT chuẩn hóa, 3 địa điểm (HCM/HN/ĐN), hình thức Remote, kỹ năng, cờ thiếu kinh nghiệm | `job_market/src/career_market/training.py` (`features()`) |
| **Tiền xử lý dữ liệu** | Xử lý khuyết thiếu bằng indicator; khử trùng repost; loại bỏ target leakage; DictVectorizer + StandardScaler | `job_market/src/career_market/preparation.py`<br>`job_market/src/career_market/training.py` |
| **Trực quan hóa** | Biểu đồ lương theo nghề, địa điểm và kinh nghiệm trên Streamlit Dashboard | `job_market/src/career_market/salary_visuals.py`<br>`job_market/pages/overview.py` |
| **Loại bài toán ML** | Hồi quy có giám sát (Supervised Regression) dự đoán lương trung điểm (VND/tháng) | `docs/rebuild/ml/SPEC.md` |
| **Thuật toán sử dụng** | **Ridge Regression** (`alpha=10`, SVD) và **Linear Regression (OLS)** | `job_market/src/career_market/training.py` |
| **Cách chia Train/Test** | Chia theo nhóm công ty (Company-held-out split) tỷ lệ 70/15/15, seed 42 | `job_market/artifacts/runs/7137291a5d16b71f1861/split.json` |
| **Metric đánh giá** | **MAE (triệu VND/tháng)** là metric chính, **RMSE** là metric phụ | `job_market/artifacts/runs/7137291a5d16b71f1861/report.json` |
| **Kết quả thực tế** | Ridge thắng: Val MAE 10,50 tr; Test MAE 21,98 tr (thắng Baseline 32,50 tr) | `docs/rebuild/ml/TRAINING_RUN-2026-10-30.md` |
| **Sản phẩm bàn giao** | File Jupyter Notebook + Ứng dụng Streamlit (`job_market/app.py`) | `salary_disclosure_classification.ipynb`<br>`bao_cao_du_an_ml.ipynb` |

---

### 4.2. Hướng dẫn mở và kiểm tra sản phẩm:
1. **File Jupyter Notebook:**
   - Mở file [`bao_cao_du_an_ml.ipynb`](file:///Users/scara2906/Downloads/ML/bao_cao_du_an_ml.ipynb) hoặc [`job_market/salary_disclosure_classification.ipynb`](file:///Users/scara2906/Downloads/ML/job_market/salary_disclosure_classification.ipynb) trong VS Code, Cursor hoặc Jupyter Lab để đọc toàn bộ báo cáo.
2. **Khởi chạy Demo Streamlit trực tiếp từ mô hình đã lưu:**
   ```sh
   make demo
   # Mở trình duyệt tại http://127.0.0.1:8502
   ```
   Ứng dụng nạp trực tiếp artifact `Ridge.joblib` để người dùng chọn vai trò, địa điểm và nhận kết quả dự đoán lương tức thì.

---
*Báo cáo được trích xuất hoàn toàn và chuẩn xác từ mã nguồn và dữ liệu thực tế của dự án `theihoz/ML`.*
